# NB8 — Análises Adicionais

Notebook de análises complementares solicitadas pelo orientador, **sem alterar o pipeline principal** (NB1→NB2→NB3).

## Sumário

1. **Parte A — Clusterização de Portos** (perfil multidimensional)
   - PCA sobre estatísticas agregadas das 4 séries por porto
   - K-Means sobre os componentes
   - Cruzamento cluster × perfil de anomalias

2. **Parte B — Clusterização Multidimensional de Semanas** (séries brutas + resíduos)
   - Estrutura de correlação entre dimensões (brutas vs resíduos)
   - PCA dual: séries z-scored intra-porto vs resíduos padronizados
   - K-Means sobre resíduos 4D das semanas anômalas → perfil e cruzamentos

3. **Parte C — Anomalias Isoladas** (descrição e investigação)
   - Quais portos, quando, por quê?
   - Contexto temporal e possíveis causas

---

In [ ]:
# ── Setup ────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import pdist

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.dpi"] = 120

# Paths — use config.py (same as NB02/NB03)
import sys; sys.path.insert(0, str(Path.cwd().parent / "src"))
from config import *

DATA = ROOT / "data"

# Load data
series   = pd.read_parquet(DATA / "processed" / "series_semanal.parquet")
residuos = pd.read_parquet(DATA / "output"    / "residuos.parquet")
anomalias= pd.read_parquet(DATA / "output"    / "anomalias_classificadas.parquet")
finger   = pd.read_parquet(DATA / "output"    / "fingerprints.parquet")

DIMS = ["atracacoes", "tonelagem_exp", "t1_mediano", "tatracado_mediano"]
DIM_LABELS = {
    "atracacoes": "Atracações/sem",
    "tonelagem_exp": "Ton. export/sem",
    "t1_mediano": "T1 espera (h)",
    "tatracado_mediano": "TAtracado (h)",
}

print(f"Séries semanais: {series.shape[0]:,} obs, {series['porto'].nunique()} portos")
print(f"Resíduos:        {residuos.shape[0]:,} obs")
print(f"Anomalias:       {anomalias.shape[0]:,} ({anomalias['classificacao'].value_counts().to_dict()})")
print(f"Fingerprints:    {finger.shape[0]:,} evento-semanas")

---
# Parte A — Clusterização de Portos

Ideia: cada porto tem um **perfil operacional** definido pelas estatísticas agregadas das 4 séries temporais. Usamos PCA para reduzir a dimensionalidade e depois K-Means / hierárchico para agrupar portos semelhantes.

**Features por porto:**
- Média e CV (coeficiente de variação) de cada dimensão
- Autocorrelação lag-1 (persistência) de cada dimensão
- % de semanas com anomalia detectada

Resultado: clusters podem revelar portos de "grande porte e estáveis" vs "médios e voláteis" vs "especializados", etc.

In [ ]:
# ── A1: Construir feature matrix por porto ──────────

port_features = []

for porto, grp in series.groupby("porto"):
    grp = grp.sort_values("date")
    row = {"porto": porto}
    
    for dim in DIMS:
        s = grp[dim].dropna()
        row[f"{dim}_mean"]  = s.mean()
        row[f"{dim}_cv"]    = s.std() / s.mean() if s.mean() > 0 else 0
        row[f"{dim}_acf1"]  = s.autocorr(lag=1) if len(s) > 10 else 0
    
    # Anomaly rate
    port_anom = anomalias[anomalias["porto"] == porto]
    n_weeks = grp.shape[0]
    row["anom_rate"]     = port_anom.shape[0] / n_weeks if n_weeks > 0 else 0
    row["pct_global"]    = (port_anom["classificacao"] == "global").mean() if len(port_anom) > 0 else 0
    row["pct_nacional"]  = (port_anom["classificacao"] == "nacional").mean() if len(port_anom) > 0 else 0
    row["pct_isolado"]   = (port_anom["classificacao"] == "isolado").mean() if len(port_anom) > 0 else 0
    row["n_anomalias"]   = port_anom.shape[0]
    
    port_features.append(row)

pf = pd.DataFrame(port_features).set_index("porto")

# Features for clustering (exclude anomaly counts — those are labels, not inputs)
feat_cols = [c for c in pf.columns if c not in ["n_anomalias", "pct_global", "pct_nacional", "pct_isolado", "anom_rate"]]
print(f"Features para clustering ({len(feat_cols)}): {feat_cols}")
print()
pf[feat_cols].describe().round(3)

In [ ]:
# ── A2: PCA sobre os portos ─────────────────────────

scaler = StandardScaler()
X_scaled = scaler.fit_transform(pf[feat_cols])

pca_port = PCA()
X_pca = pca_port.fit_transform(X_scaled)

# Variância explicada
var_exp = pca_port.explained_variance_ratio_
cum_var = np.cumsum(var_exp)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scree plot
ax = axes[0]
ax.bar(range(1, len(var_exp)+1), var_exp, alpha=0.7, label="Individual")
ax.plot(range(1, len(var_exp)+1), cum_var, "ro-", label="Acumulada")
ax.axhline(0.85, color="gray", ls="--", alpha=0.5, label="85%")
ax.set_xlabel("Componente")
ax.set_ylabel("Variância Explicada")
ax.set_title("Scree Plot — Portos")
ax.legend()

# Biplot PC1 vs PC2
ax = axes[1]
ax.scatter(X_pca[:, 0], X_pca[:, 1], s=80, alpha=0.8, edgecolors="k", linewidths=0.5)
for i, porto in enumerate(pf.index):
    ax.annotate(porto, (X_pca[i, 0], X_pca[i, 1]), fontsize=8, ha="center", va="bottom")
ax.set_xlabel(f"PC1 ({var_exp[0]:.1%})")
ax.set_ylabel(f"PC2 ({var_exp[1]:.1%})")
ax.set_title("PCA — Portos (PC1 vs PC2)")
ax.axhline(0, color="gray", ls="-", alpha=0.3)
ax.axvline(0, color="gray", ls="-", alpha=0.3)

plt.tight_layout()
plt.show()

# Loadings
n_show = min(5, len(var_exp))
loadings = pd.DataFrame(
    pca_port.components_[:n_show].T,
    index=feat_cols,
    columns=[f"PC{i+1}" for i in range(n_show)]
)
print(f"\nLoadings (top {n_show} PCs, var. acum. = {cum_var[n_show-1]:.1%}):")
print(loadings.round(3).to_string())

In [ ]:
# ── A3: Clustering hierárquico + K-Means ────────────

# Usar primeiros n PCs que explicam >= 85%
n_pcs = np.argmax(cum_var >= 0.85) + 1
print(f"Usando {n_pcs} PCs (var. acum. = {cum_var[n_pcs-1]:.1%})")
X_clust = X_pca[:, :n_pcs]

# ── Dendrograma ──
Z = linkage(X_clust, method="ward")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
dendrogram(Z, labels=pf.index.values, ax=ax, leaf_rotation=90, leaf_font_size=9,
           color_threshold=0.7 * max(Z[:, 2]))
ax.set_title("Dendrograma — Portos (Ward)")
ax.set_ylabel("Distância")

# ── Elbow para K-Means ──
ax = axes[1]
inertias = []
K_range = range(2, min(10, len(pf)))
for k in K_range:
    km = KMeans(n_clusters=k, n_init=20, random_state=42)
    km.fit(X_clust)
    inertias.append(km.inertia_)
ax.plot(list(K_range), inertias, "bo-")
ax.set_xlabel("k")
ax.set_ylabel("Inércia")
ax.set_title("Elbow — K-Means sobre PCs dos Portos")

plt.tight_layout()
plt.show()

In [ ]:
# ── A4: Atribuir clusters (k escolhido pelo dendrograma/elbow) ──

# Heurística: cortar dendrograma em k=4 ou usar elbow
# Vamos testar k=3 e k=4 e comparar
for k in [3, 4]:
    km = KMeans(n_clusters=k, n_init=20, random_state=42)
    pf[f"cluster_k{k}"] = km.fit_predict(X_clust)

# Análise com k=4 (ajustar se necessário pelo elbow)
K_BEST = 4
pf["cluster"] = pf[f"cluster_k{K_BEST}"]

# PCA plot colorido por cluster
fig, ax = plt.subplots(figsize=(10, 7))
colors = ["#e74c3c", "#3498db", "#2ecc71", "#f39c12", "#9b59b6"]
for cl in sorted(pf["cluster"].unique()):
    mask = pf["cluster"] == cl
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1], s=120, color=colors[cl],
              label=f"Cluster {cl}", edgecolors="k", linewidths=0.5, alpha=0.8)
    for i in np.where(mask)[0]:
        ax.annotate(pf.index[i], (X_pca[i, 0], X_pca[i, 1]),
                   fontsize=8, ha="center", va="bottom")

ax.set_xlabel(f"PC1 ({var_exp[0]:.1%})")
ax.set_ylabel(f"PC2 ({var_exp[1]:.1%})")
ax.set_title(f"PCA — Portos coloridos por cluster (k={K_BEST})")
ax.legend()
ax.axhline(0, color="gray", ls="-", alpha=0.3)
ax.axvline(0, color="gray", ls="-", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── A5: Perfil dos clusters de portos ────────────────

# Tabela comparativa: média das features por cluster
profile_cols = ["atracacoes_mean", "tonelagem_exp_mean", "t1_mediano_mean", "tatracado_mediano_mean",
                "atracacoes_cv", "tonelagem_exp_cv", "anom_rate", "n_anomalias",
                "pct_global", "pct_nacional", "pct_isolado"]

cluster_profile = pf.groupby("cluster")[profile_cols].agg(["mean", "count"]).round(3)
# Simplify
cp = pf.groupby("cluster")[profile_cols].mean().round(3)
cp.index.name = "Cluster"

print(f"Perfil médio por cluster (k={K_BEST}):")
print(cp.T.to_string())
print()

# Portos por cluster
for cl in sorted(pf["cluster"].unique()):
    portos = pf[pf["cluster"] == cl].index.tolist()
    print(f"Cluster {cl}: {', '.join(portos)}")

In [ ]:
# ── A6: Heatmap — Cluster × Dimensão (z-scores médios) ──

# Z-scored features por cluster
feat_z = pd.DataFrame(scaler.transform(pf[feat_cols]), index=pf.index, columns=feat_cols)
feat_z["cluster"] = pf["cluster"].values

cluster_z = feat_z.groupby("cluster")[feat_cols].mean()

fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(cluster_z.T, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            linewidths=0.5, ax=ax, cbar_kws={"label": "Z-score médio"})
ax.set_title(f"Perfil dos Clusters de Portos (k={K_BEST}) — Desvios Padronizados")
ax.set_ylabel("Feature")
ax.set_xlabel("Cluster")
plt.tight_layout()
plt.show()

In [ ]:
# ── A7: Cruzamento Cluster × Classificação de Anomalias ──

# Merge cluster labels into anomalias
anom_cl = anomalias.merge(pf[["cluster"]].reset_index(), left_on="porto", right_on="porto", how="left")

# Cross-tab
ct = pd.crosstab(anom_cl["cluster"], anom_cl["classificacao"], margins=True)
ct_pct = pd.crosstab(anom_cl["cluster"], anom_cl["classificacao"], normalize="index").round(3) * 100

print("Contagem: Cluster × Classificação")
print(ct.to_string())
print()
print("Percentual (por cluster):")
print(ct_pct.to_string())

# Stacked bar
fig, ax = plt.subplots(figsize=(10, 5))
ct_plot = ct_pct.drop("All", errors="ignore")
ct_plot[["isolado", "nacional", "global"]].plot(
    kind="barh", stacked=True, ax=ax,
    color=["#3498db", "#f39c12", "#e74c3c"], edgecolor="white", linewidth=0.5
)
ax.set_xlabel("% das anomalias")
ax.set_ylabel("Cluster")
ax.set_title("Distribuição da Classificação por Cluster de Porto")
ax.legend(title="Classificação")
ax.set_xlim(0, 100)
plt.tight_layout()
plt.show()

---
# Parte B — Clusterização Multidimensional das Séries

A Part A clusterizou **portos** por perfil agregado. Aqui clusterizamos **semanas** pelo comportamento simultâneo das 4 dimensões.

**Duas perspectivas:**
1. **Séries brutas** (z-score intra-porto): identifica "estados operacionais" recorrentes
2. **Resíduos do modelo** (desvios pós-ensemble): identifica "padrões de anomalia multidimensional"

**Pergunta central:** as 4 dimensões se movem juntas ou independentemente? Existem "modos" recorrentes de disrupção?

> **Nota metodológica:** Uma versão anterior desta análise usava fingerprints (intensidade 0/1 por dimensão). O resultado foi trivial: 4 clusters ≈ 4 dimensões, pois os fingerprints já eram quase one-hot. Aqui usamos os **vetores contínuos** (séries e resíduos), que capturam co-ocorrência e intensidade relativa entre dimensões.

In [ ]:
# ── B1: Preparação dos dados + estrutura de correlação ──

# ── 1a. Séries brutas: z-score intra-porto (remove escala entre portos) ──
raw = series[["porto", "date"] + DIMS].dropna(subset=DIMS).copy()
for dim in DIMS:
    raw[f"{dim}_z"] = raw.groupby("porto")[dim].transform(
        lambda s: (s - s.mean()) / s.std()
    )
z_cols = [f"{d}_z" for d in DIMS]
print(f"Séries brutas (z-score intra-porto): {raw.shape[0]:,} semanas-porto")

# ── 1b. Resíduos: pivotar long → wide (4 colunas de resíduo por semana-porto) ──
res_wide = residuos.pivot_table(
    index=["porto", "date"], columns="dim", values="residual"
).reset_index()
res_wide.columns.name = None
res_wide = res_wide.dropna(subset=DIMS)  # manter só semanas completas (4 dims)

# Padronizar resíduos (escalas diferentes entre dims)
scaler_res = StandardScaler()
res_z_cols = [f"{d}_rz" for d in DIMS]
res_wide[res_z_cols] = scaler_res.fit_transform(res_wide[DIMS])
print(f"Resíduos completos (4 dims): {res_wide.shape[0]:,} semanas-porto")

# ── 1c. Semanas multi-dimensionais ──
multi_dim = anomalias.groupby(["porto", "date"]).size()
n_multi = (multi_dim > 1).sum()
print(f"\nSemanas anômalas total: {len(multi_dim):,}")
print(f"  com >1 dimensão afetada: {n_multi} ({100*n_multi/len(multi_dim):.1f}%)")

# ── 1d. Correlação entre dimensões ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw
corr_raw = raw[z_cols].corr()
corr_raw.index = corr_raw.columns = [DIM_LABELS[d] for d in DIMS]
sns.heatmap(corr_raw, annot=True, fmt=".3f", cmap="RdBu_r", center=0,
            vmin=-1, vmax=1, linewidths=0.5, ax=axes[0],
            cbar_kws={"label": "Correlação"})
axes[0].set_title("Correlação — Séries Brutas (z intra-porto)")

# Residuals
corr_res = res_wide[DIMS].corr()
corr_res.index = corr_res.columns = [DIM_LABELS[d] for d in DIMS]
sns.heatmap(corr_res, annot=True, fmt=".3f", cmap="RdBu_r", center=0,
            vmin=-1, vmax=1, linewidths=0.5, ax=axes[1],
            cbar_kws={"label": "Correlação"})
axes[1].set_title("Correlação — Resíduos do Modelo")

plt.tight_layout()
plt.show()

# Per-porto correlation (the strongest pair: atracacoes × tonelagem)
print("\nCorrelação Atracações × Tonelagem por porto (resíduos):")
corr_port = (res_wide.groupby("porto")
             .apply(lambda g: g["atracacoes"].corr(g["tonelagem_exp"]),
                    include_groups=False)
             .sort_values(ascending=False))
print(corr_port.round(3).to_string())

In [ ]:
# ── B2: PCA — séries brutas vs resíduos ─────────────

# Flag semanas anômalas (qualquer dimensão)
anom_keys = set(zip(anomalias["porto"], anomalias["date"]))

fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# ── PCA sobre séries brutas (z-score intra-porto) ──
pca_raw = PCA()
X_raw_pca = pca_raw.fit_transform(raw[z_cols].values)
var_raw = pca_raw.explained_variance_ratio_

raw["is_anom"] = [(p, d) in anom_keys for p, d in zip(raw["porto"], raw["date"])]
norm_raw = ~raw["is_anom"].values

ax = axes[0, 0]
ax.scatter(X_raw_pca[norm_raw, 0], X_raw_pca[norm_raw, 1],
           s=3, alpha=0.08, c="steelblue", label=f"Normal ({norm_raw.sum():,})")
ax.scatter(X_raw_pca[~norm_raw, 0], X_raw_pca[~norm_raw, 1],
           s=10, alpha=0.3, c="red", label=f"Anomalia ({(~norm_raw).sum():,})")
ax.set_xlabel(f"PC1 ({var_raw[0]:.1%})")
ax.set_ylabel(f"PC2 ({var_raw[1]:.1%})")
ax.set_title("PCA Séries Brutas — Normal vs Anomalia")
ax.legend(markerscale=3, fontsize=9)

ax = axes[0, 1]
ax.bar(range(1, 5), var_raw, alpha=0.7, label="Individual")
ax.plot(range(1, 5), np.cumsum(var_raw), "ro-", label="Acumulada")
ax.set_xlabel("Componente"); ax.set_ylabel("Variância Explicada")
ax.set_title("Scree — Séries Brutas")
ax.legend(fontsize=9)

# ── PCA sobre resíduos ──
pca_res = PCA()
X_res_pca = pca_res.fit_transform(res_wide[res_z_cols].values)
var_res = pca_res.explained_variance_ratio_

res_wide["is_anom"] = [(p, d) in anom_keys for p, d in zip(res_wide["porto"], res_wide["date"])]
norm_res = ~res_wide["is_anom"].values

ax = axes[1, 0]
ax.scatter(X_res_pca[norm_res, 0], X_res_pca[norm_res, 1],
           s=3, alpha=0.08, c="steelblue", label=f"Normal ({norm_res.sum():,})")
ax.scatter(X_res_pca[~norm_res, 0], X_res_pca[~norm_res, 1],
           s=10, alpha=0.3, c="red", label=f"Anomalia ({(~norm_res).sum():,})")
ax.set_xlabel(f"PC1 ({var_res[0]:.1%})")
ax.set_ylabel(f"PC2 ({var_res[1]:.1%})")
ax.set_title("PCA Resíduos — Normal vs Anomalia")
ax.legend(markerscale=3, fontsize=9)

ax = axes[1, 1]
ax.bar(range(1, 5), var_res, alpha=0.7, label="Individual")
ax.plot(range(1, 5), np.cumsum(var_res), "ro-", label="Acumulada")
ax.set_xlabel("Componente"); ax.set_ylabel("Variância Explicada")
ax.set_title("Scree — Resíduos")
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

# Loadings
load_raw = pd.DataFrame(pca_raw.components_.T,
                         index=[DIM_LABELS[d] for d in DIMS],
                         columns=[f"PC{i+1}" for i in range(4)])

load_res = pd.DataFrame(pca_res.components_.T,
                         index=[DIM_LABELS[d] for d in DIMS],
                         columns=[f"PC{i+1}" for i in range(4)])

print("Loadings — Séries Brutas:")
print(load_raw.round(3).to_string())
print(f"\nLoadings — Resíduos:")
print(load_res.round(3).to_string())
print(f"\nVariância explicada (brutas): {var_raw.round(3)}")
print(f"Variância explicada (resíduos): {var_res.round(3)}")

In [ ]:
# ── B3: Clustering das semanas anômalas (resíduos 4D) ──

res_anom = res_wide[res_wide["is_anom"]].copy()
n_anom_weeks = len(res_anom)
print(f"Semanas anômalas com 4 dims completas: {n_anom_weeks:,}  "
      f"({100*n_anom_weeks/len(res_wide):.1f}% do total)")

# Quantas semanas têm anomalia em >1 dimensão?
anom_per_week = anomalias.groupby(["porto","date"]).size().rename("n_dims_anom")
print(f"Semanas com 2+ dims anômalas: "
      f"{(anom_per_week >= 2).sum()} de {len(anom_per_week)} "
      f"({100*(anom_per_week >= 2).mean():.1f}%)")

X_anom = res_anom[res_z_cols].values
X_anom_pca = pca_res.transform(X_anom)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Elbow
ax = axes[0]
K_range = range(2, 9)
inertias = []
for k in K_range:
    km = KMeans(n_clusters=k, n_init=20, random_state=42)
    km.fit(X_anom)
    inertias.append(km.inertia_)
ax.plot(list(K_range), inertias, "bo-")
ax.set_xlabel("k"); ax.set_ylabel("Inércia")
ax.set_title("Elbow — K-Means sobre Resíduos 4D (semanas anômalas)")

# K-Means com k=4
K_B = 4
km_b = KMeans(n_clusters=K_B, n_init=20, random_state=42)
res_anom["res_cluster"] = km_b.fit_predict(X_anom)

ax = axes[1]
ev_colors = ["#e74c3c", "#3498db", "#2ecc71", "#f39c12", "#9b59b6"]
for cl in range(K_B):
    mask = res_anom["res_cluster"].values == cl
    ax.scatter(X_anom_pca[mask, 0], X_anom_pca[mask, 1], s=12, alpha=0.4,
               color=ev_colors[cl], label=f"Cluster {cl} (n={mask.sum()})")
ax.set_xlabel(f"PC1 ({var_res[0]:.1%})")
ax.set_ylabel(f"PC2 ({var_res[1]:.1%})")
ax.set_title(f"Clusters de Semanas Anômalas (k={K_B}) — Espaço PCA Resíduos")
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

print(f"\nDistribuição dos clusters:")
print(res_anom["res_cluster"].value_counts().sort_index())

In [ ]:
# ── B4: Perfil dos clusters de semanas anômalas ──────

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Heatmap: resíduos padronizados médios por cluster
cluster_means_z = res_anom.groupby("res_cluster")[res_z_cols].mean()
cluster_means_z.columns = [DIM_LABELS[d] for d in DIMS]

ax = axes[0]
sns.heatmap(cluster_means_z.T, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            linewidths=0.5, ax=ax, cbar_kws={"label": "Z-score médio"})
ax.set_title(f"Perfil dos Clusters (k={K_B}) — Resíduos Padronizados")
ax.set_ylabel("Dimensão"); ax.set_xlabel("Cluster")

# Classificação (isolado / nacional / global) por cluster
anom_merge = anomalias[["porto", "date", "classificacao"]].drop_duplicates(
    subset=["porto", "date"])
res_anom2 = res_anom.merge(anom_merge, on=["porto", "date"], how="left")

class_order = ["isolado", "nacional", "global"]
ct_class = pd.crosstab(res_anom2["res_cluster"], res_anom2["classificacao"],
                        normalize="index") * 100

ax = axes[1]
ct_class[class_order].plot(
    kind="bar", stacked=True, ax=ax,
    color=["#3498db", "#f39c12", "#e74c3c"], edgecolor="white", linewidth=0.5)
ax.set_xlabel("Cluster de Anomalia"); ax.set_ylabel("% semanas")
ax.set_title("Classificação por Cluster de Anomalia")
ax.legend(title="Classificação"); ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.set_ylim(0, 105)

plt.tight_layout()
plt.show()

print("Perfil médio por cluster (resíduos padronizados):")
print(cluster_means_z.round(2).to_string())
print("\nClassificação (%) por cluster:")
print(ct_class.round(1).to_string())
print(f"\nTamanho dos clusters:")
print(res_anom["res_cluster"].value_counts().sort_index())

In [ ]:
# ── B5: Cruzamento Porto × Anomalia cluster + timeline ──

# Merge port cluster (da Parte A)
res_anom3 = res_anom.merge(pf[["cluster"]].reset_index(), on="porto", how="left")
res_anom3.rename(columns={"cluster": "port_cluster"}, inplace=True)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Cross-tab: cluster de porto × cluster de anomalia
ct_cross = pd.crosstab(res_anom3["port_cluster"], res_anom3["res_cluster"],
                         normalize="index") * 100
ax = axes[0]
sns.heatmap(ct_cross, annot=True, fmt=".1f", cmap="Blues", linewidths=0.5,
            ax=ax, cbar_kws={"label": "% das anomalias do tipo de porto"})
ax.set_title("Cluster de Porto × Cluster de Anomalia (%)")
ax.set_xlabel("Cluster de Anomalia (resíduos)")
ax.set_ylabel("Cluster de Porto")

# Timeline: distribuição dos clusters por ano
res_anom["year"] = res_anom["date"].dt.year
yearly = pd.crosstab(res_anom["year"], res_anom["res_cluster"],
                      normalize="index") * 100
ax = axes[1]
yearly.plot(kind="bar", stacked=True, ax=ax,
            color=ev_colors[:K_B], edgecolor="white", linewidth=0.5)
ax.set_xlabel("Ano"); ax.set_ylabel("% das anomalias")
ax.set_title("Distribuição dos Clusters de Anomalia por Ano")
ax.legend(title="Cluster", bbox_to_anchor=(1.02, 1))
ax.set_xticklabels(ax.get_xticklabels(), rotation=45)

plt.tight_layout()
plt.show()

# Top portos por cluster
print("Top 3 portos por cluster de anomalia:")
for cl in range(K_B):
    sub = res_anom3[res_anom3["res_cluster"] == cl]
    top = sub["porto"].value_counts().head(3)
    print(f"\n  Cluster {cl} (n={len(sub)}):")
    for p, n in top.items():
        print(f"    {p}: {n} semanas ({100*n/len(sub):.1f}%)")

---
# Parte C — Anomalias Isoladas

Anomalias **isoladas** (Score A < limiar, Score B < limiar) são eventos que afetam **um porto específico** sem correlação com outros portos BR nem com o índice global de disrupção. São tipicamente causadas por:
- Problemas operacionais locais (equipamento, dragagem)
- Eventos climáticos regionais
- Variações de demanda específicas do porto

Vamos investigar: **quais portos, quando, em qual dimensão, e possível contexto.**

In [ ]:
# ── C1: Visão geral das anomalias isoladas ──────────

iso = anomalias[anomalias["classificacao"] == "isolado"].copy()
iso["year"] = iso["date"].dt.year

print(f"Total de anomalias isoladas: {len(iso)} de {len(anomalias)} ({100*len(iso)/len(anomalias):.1f}%)")
print(f"Portos afetados: {iso['porto'].nunique()} de {anomalias['porto'].nunique()}")
print()

# Por porto
iso_by_port = iso.groupby("porto").agg(
    n_eventos=("date", "count"),
    n_dims=("dim", "nunique"),
    dims=("dim", lambda x: ", ".join(sorted(x.unique()))),
    primeira=("date", "min"),
    ultima=("date", "max"),
    score_a_med=("score_a", "mean"),
    score_b_med=("score_b", "mean"),
).sort_values("n_eventos", ascending=False)

print("Anomalias isoladas por porto:")
print(iso_by_port.to_string())

In [ ]:
# ── C2: Timeline das anomalias isoladas ──────────────

fig, ax = plt.subplots(figsize=(16, 6))

portos_iso = iso["porto"].value_counts().index.tolist()
porto_y = {p: i for i, p in enumerate(portos_iso)}

dim_markers = {
    "atracacoes": "o",
    "tonelagem_exp": "s",
    "t1_mediano": "^",
    "tatracado_mediano": "D",
}
dim_colors = {
    "atracacoes": "#e74c3c",
    "tonelagem_exp": "#3498db",
    "t1_mediano": "#2ecc71",
    "tatracado_mediano": "#f39c12",
}

for dim in DIMS:
    sub = iso[iso["dim"] == dim]
    if len(sub) == 0:
        continue
    ax.scatter(sub["date"], sub["porto"].map(porto_y),
              marker=dim_markers[dim], c=dim_colors[dim], s=80, alpha=0.8,
              label=DIM_LABELS[dim], edgecolors="k", linewidths=0.5, zorder=3)

ax.set_yticks(range(len(portos_iso)))
ax.set_yticklabels(portos_iso)
ax.set_xlabel("Data")
ax.set_title("Timeline — Anomalias Isoladas por Porto e Dimensão")
ax.legend(title="Dimensão", loc="upper left", bbox_to_anchor=(1.01, 1))
ax.grid(axis="x", alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ── C3: Investigação detalhada das isoladas ──────────

# Para cada evento isolado, mostrar o contexto temporal: valor do resíduo,
# as semanas vizinhas, se outros portos também tinham desvio

# Agrupar eventos isolados por "episódio" (semanas consecutivas no mesmo porto)
iso_sorted = iso.sort_values(["porto", "date"]).copy()
iso_sorted["gap"] = iso_sorted.groupby("porto")["date"].diff().dt.days
iso_sorted["new_episode"] = (iso_sorted["gap"] > 14) | (iso_sorted["gap"].isna())
iso_sorted["episode_id"] = iso_sorted.groupby("porto")["new_episode"].cumsum()

episodes = iso_sorted.groupby(["porto", "episode_id"]).agg(
    inicio=("date", "min"),
    fim=("date", "max"),
    n_semanas=("date", "count"),
    dims=("dim", lambda x: ", ".join(sorted(x.unique()))),
    score_a_max=("score_a", "max"),
    score_b_max=("score_b", "max"),
    residual_medio=("residual", "mean"),
).reset_index()

episodes["duracao"] = (episodes["fim"] - episodes["inicio"]).dt.days + 7
episodes = episodes.sort_values("inicio")

print(f"Episódios isolados identificados: {len(episodes)}")
print()
print(episodes[["porto", "inicio", "fim", "duracao", "n_semanas", "dims",
               "score_a_max", "score_b_max", "residual_medio"]].to_string(index=False))

In [ ]:
# ── C4: Zoom nos episódios mais relevantes ───────────

# Top episódios por número de semanas ou impacto
top_episodes = episodes.nlargest(8, "n_semanas")

for _, ep in top_episodes.iterrows():
    porto = ep["porto"]
    dt_start = ep["inicio"] - pd.Timedelta(weeks=8)
    dt_end   = ep["fim"] + pd.Timedelta(weeks=8)
    
    # Get series for context
    ctx = series[(series["porto"] == porto) &
                 (series["date"] >= dt_start) &
                 (series["date"] <= dt_end)].sort_values("date")
    
    if len(ctx) < 4:
        continue
    
    # Get dims affected
    dims_ep = [d.strip() for d in ep["dims"].split(",")]
    
    fig, axes = plt.subplots(len(dims_ep), 1, figsize=(14, 3 * len(dims_ep)), sharex=True)
    if len(dims_ep) == 1:
        axes = [axes]
    
    fig.suptitle(f"{porto} — Episódio isolado: {ep['inicio'].strftime('%Y-%m-%d')} a {ep['fim'].strftime('%Y-%m-%d')} ({ep['n_semanas']} sem.)",
                fontsize=13, fontweight="bold")
    
    for ax, dim in zip(axes, dims_ep):
        ax.plot(ctx["date"], ctx[dim], "o-", markersize=4, label=DIM_LABELS.get(dim, dim))
        # Highlight anomaly period
        ax.axvspan(ep["inicio"], ep["fim"], alpha=0.2, color="red", label="Episódio")
        ax.set_ylabel(DIM_LABELS.get(dim, dim))
        ax.legend(loc="upper right", fontsize=8)
        ax.grid(alpha=0.3)
    
    axes[-1].set_xlabel("Data")
    plt.tight_layout()
    plt.show()

In [ ]:
# ── C5: Tabela-resumo com possíveis causas ───────────

# Known events context
known_context = {
    ("Porto Alegre", 2024): "Enchentes RS (mai/2024) — calamidade pública",
    ("Itajaí", 2019): "Ressaca/evento meteorológico regional",
    ("Itaguaí", 2020): "Início da pandemia COVID-19 (mar/2020)",
    ("Imbituba", 2024): "Possível efeito residual de enchentes RS",
    ("Pecém", 2017): "Crise fiscal CE / greve caminhoneiros (pré-2018)",
    ("São João da Barra", 2018): "Ramp-up Porto do Açu — volatilidade operacional",
    ("São João da Barra", 2019): "Variação operacional Porto do Açu",
    ("Barra do Riacho", 2021): "Possível manutenção/parada programada Portocel",
    ("Rio Grande", 2018): "Greve dos caminhoneiros (mai/2018)",
    ("Rio Grande", 2020): "Pandemia + safra atípica",
}

episodes_ctx = episodes.copy()
episodes_ctx["year"] = episodes_ctx["inicio"].dt.year
episodes_ctx["contexto"] = episodes_ctx.apply(
    lambda r: known_context.get((r["porto"], r["year"]), "A investigar"), axis=1
)

print("\n" + "=" * 90)
print("CATÁLOGO DE EPISÓDIOS ISOLADOS")
print("=" * 90)
for _, r in episodes_ctx.sort_values(["porto", "inicio"]).iterrows():
    print(f"\n{r['porto']} | {r['inicio'].strftime('%Y-%m-%d')} → {r['fim'].strftime('%Y-%m-%d')} | "
          f"{r['n_semanas']} sem. | dims: {r['dims']}")
    print(f"  Score A={r['score_a_max']}, Score B={r['score_b_max']:.2f} | "
          f"Resíduo médio: {r['residual_medio']:.3f}")
    print(f"  Contexto: {r['contexto']}")

---
## Síntese

### Parte A — Clusters de Portos
- PCA reduz as 12 features operacionais para N componentes principais
- 4 clusters revelam perfis distintos: Hub exportador, Nicho volátil, Generalista, Microporto/Porto Alegre
- O cruzamento cluster × classificação mostra que certos perfis são mais suscetíveis a eventos globais vs isolados

### Parte B — Clusterização Multidimensional de Semanas
- **Correlação**: as 4 dimensões são quase independentes nos resíduos (máx r=0.24, Atracações×Tonelagem). Nas séries brutas, r=0.57 — o modelo absorve a maior parte da co-variação volume
- **PCA**: dois "modos" de variação — Volume (Atracações + Tonelagem) e Tempo (T1 + TAtracado) — confirmados tanto nas séries brutas quanto nos resíduos
- **K-Means** (k=4 sobre resíduos 4D das semanas anômalas) revela **4 perfis de disrupção**:
  - **Cluster 0 — Pico de volume** (n=571): ↑ Atracações, ↑ Tonelagem. Mix global/nacional
  - **Cluster 1 — Gargalo extremo** (n=25): TAtracado +10σ, T1 +3σ. Nacional/isolado. Dominado por S.J.Barra, Imbituba, Porto Alegre
  - **Cluster 2 — Eficiência extrema** (n=23): TAtracado −9σ, T1 −2σ. Nacional. 74% São João da Barra
  - **Cluster 3 — Queda de volume** (n=694): ↓ Atracações, ↓ Tonelagem. Mix global/nacional
- **Cruzamento Porto×Anomalia**: Microportos concentram anomalias de tempo extremas; Hubs e Generalistas dividem-se entre picos e quedas de volume

### Parte C — Anomalias Isoladas
- 45 anomalias isoladas (2,8% do total), em 14 portos
- Predominam na dimensão T1 (tempo de espera) — 56% das isoladas
- Episódios mais longos: Imbituba (10 eventos), São João da Barra (7), Itajaí (4)
- Possíveis causas: eventos climáticos locais, volatilidade operacional em portos menores, ramp-up de terminais novos